---
title: "The Agent Harness: Building Claude Code from Scratch"
description: "A build-first course on everything between a language model and a system trusted with a repository: streaming client, tool protocol, agent loop, context management, permissions, hooks, sub-agents, MCP, and evaluation."
categories: [agents, engineering]
---


A raw language model cannot act. It has no persistent state, no effect on the world, and no access to ground truth: every request begins and ends in the same context window. Everything between the weights and a system you would trust with a repository is the **harness**: the client that talks to the API, the tools the model can call, the loop that turns responses into actions, the machinery that manages context, instructions, permissions, and hooks, and the evaluation that makes claims about the whole thing falsifiable.

This course builds that harness from scratch, one mechanism per week, into the Python package `projects/agent-harness/`. The working thesis, tested rather than asserted: a Claude-Code-class coding agent is a thin loop plus twenty good decisions, and every one of those decisions can be built, broken, and measured in isolation. Weeks 1 through 5 build the minimum loop that acts. Weeks 6 through 9 make it governable. Weeks 10 and 11 scale it out. Week 12 makes the whole claim falsifiable with an ablated-harness scorecard.

**Design rule:** every layer that adds agency must also add a mechanism that measures and constrains its new failure modes. A layer that only adds capability is not finished.


## The learning path

Each chapter ships one or two mechanisms into the library and runs one controlled experiment on them. The experiment column is not decoration: the measured delta is the chapter's actual result.

| Ch | Chapter | Mechanism merged into `agent_harness` | Main experiment |
|----|---------|----------------------------------------|-----------------|
| 01 | Model versus Harness | nothing yet — two deliberately naive loops | task success with and without the loop |
| 02 | The Streaming Client | `client.py` — transport, retries, token ledger | retry storms and accounting drift |
| 03 | The Tool Protocol | `tools/base.py` — `Tool`, `ToolRegistry` | description ablations and error-recovery rates |
| 04 | Core Coding Tools | `tools/files.py`, `tools/shell.py` | edit success under ambiguity perturbations |
| 05 | The Agent Loop | `session.py`, `agent.py`, `events.py` | turn-budget curves and error cascades |
| 06 | Context Management | `context.py`, `compaction.py`, `prompts.py` | success under context pressure; compaction loss |
| 07 | Instructions and Memory | `config.py`, `tools/memory.py` | instruction compliance vs placement |
| 08 | Permissions and Sandboxing | `safety.py`, `sandbox.py` | false-block vs caught-dangerous on a labeled corpus |
| 09 | Hooks | `hooks.py` | style-gate precision via hooks |
| 10 | Sub-Agents and Patterns | `tools/task.py`, `subagents.py` | isolated vs shared context on research tasks |
| 11 | Model Context Protocol | `mcp/client.py`, `mcp/bridge.py` | third-party server integration; injection surface |
| 12 | Capstone | `eval/` — task suite, trajectory store, judge, scorecard | ablated-harness scorecard on held-out repos |

The four parts group the arc: **Part I** (01–03) exposes the engine, **Part II** (04–05) makes it a coding agent, **Part III** (06–09) makes it controllable, **Part IV** (10–12) scales it out and then proves it.


## Prerequisites

Tooling:

- Python 3.13+ with comfort in `async`/`await`; [pytest](https://docs.pytest.org/) for the weekly regression tests.
- A frontier model behind an OpenAI-compatible API. The loop logic is provider-independent; wire-format differences (Anthropic's in particular) appear in one sidebar in Week 2.
- [Git](https://git-scm.com/) — the agent operates on real repositories from Week 4 on.
- HTTP basics. Server-Sent Events are parsed by hand once, in Week 2, and never taken on faith again.
- Docker appears only in Week 8, as one sandboxing backend among others.

Assumed knowledge (from the language-model course, or equivalent):

- Chat templates and role structure, including tool roles.
- Tool-call schema validity and the three conditions of a correct call: the right tool, valid arguments, and calling being the right decision at all.
- Evaluation discipline: held-out sets, regression suites, seeds.
- The reliability vocabulary — specification error, proxy optimization, correction failure, distribution shift — redefined briefly in Week 1 as it becomes observable in your own runs.


## Important notes

- **The library is the course.** `projects/agent-harness/` is a uv workspace member that accumulates one or two modules per week. Each chapter's notebook ends with that week's mechanism merged into the package and exercised through imports rather than copy-paste; regression tests land in the same week as the mechanism they guard. Reading the chapters in order reconstructs the package.
- **No GPU anywhere.** The subject is the harness, not the weights.
- **The reliability lens runs through everything.** Every week asks three questions about its mechanism: what does it assume about model behavior, and when is that assumption false; what new damage can this layer cause that the previous layer could not; how would we detect the mechanism failing silently. Discussion of negative results is expected — most harness tuning is negative results.
- **Assessment.** Each week pairs a protocol or threat analysis (what contract is assumed, what breaks it) with a from-scratch implementation, a controlled experiment, and a written interpretation naming the reliability terms observed in your own runs. From Week 4 on, add one regression test guarding a failure mode introduced that week.
